<a href="https://colab.research.google.com/github/MinsooKwak/RAG/blob/main/test/prompt/basic_prompt_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- langchain hub 사이트 : https://smith.langchain.com/hub

In [ ]:
#!pip install langchain openai langchain_community

### Lazy Prompt > Prompt

- https://smith.langchain.com/hub/hardkothari/prompt-maker
- 변수 2개 (필수)
  - task : 어떤 걸 할지
  - lazy_prompt : 내가 입력한 프롬프트 (개선할 프롬프트)

In [24]:
import os
import openai
from openai import OpenAI
from config import OPEN_AI_API_KEY
from langchain.chat_models import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import load_prompt
from langchain_core.output_parsers import StrOutputParser

from langchain import hub

os.environ["OPENAI_API_KEY"] = OPEN_AI_API_KEY

client = OpenAI(api_key=OPEN_AI_API_KEY)

In [18]:
def create_chain(prompt_file_path, task=None, lazy_prompt=None, selected_text=None):
  # prompt 적용
  prompt = load_prompt(prompt_file_path, encoding='utf-8')

  # gpt
  llm = ChatOpenAI(model_name='gpt-4o', temperature=0)

  # 출력 파서
  output_parser = StrOutputParser()

  # 체인
  chain = prompt | llm | output_parser

  inputs = {
    "task": task,
    "lazy_prompt":lazy_prompt,
    "selected_text":"\n".join(selected_text)
  }

  #result
  result = chain.invoke(inputs)
  return result

In [15]:
# 샘플 text 리스트 (RAG 추출해서 몇 개 뽑아냈다 했을 때 등)
selected_text = [
    "이 연구는 당뇨병 환자의 혈당 조절을 개선하기 위한 새로운 치료 방법을 제안합니다. 새로운 약물 X는 기존 약물보다 혈당 변동성을 30% 감소시켰으며, 부작용 발생률도 낮았습니다.",
    "임상시험은 6개월간 진행되었으며, 500명의 환자가 참여했습니다. 환자군은 기존 치료제와 새로운 약물을 사용한 그룹으로 나뉘어 효과를 비교했습니다.",
    "새로운 약물 X는 장기적인 혈당 관리에 효과적이며, 특히 인슐린 저항성이 높은 환자들에게 유의미한 개선 효과를 보였습니다."
]

In [16]:
task1 = "요약"
lazy_prompt1 = '''
당신은 논문의 내용을 요약하는 전문가입니다.
당신의 임무는 전문적인 요약을 하는 것입니다.
selected_text 리스트의 내용을 반영해 요약문을 작성해주세요.
'''

- deepL 활용해 번역
  - https://www.deepl.com/ko/your-account/subscription

In [21]:
#!pip install -qU deepl

In [28]:
import requests

# api 키
deepl_api_key = "YOUR_API_KEY"

def translate_text(text, source_lang="KO", target_lang="EN", api_key=deepl_api_key):
    url = "https://api-free.deepl.com/v2/translate"
    data = {
        "text": text,
        "source_lang": source_lang,
        "target_lang": target_lang,
        "auth_key": api_key
    }
    response = requests.post(url, data=data)
    if response.status_code == 200:
        return response.json()["translations"][0]["text"]
    else:
        print(f"Error: {response.status_code}, {response.text}")
        return None
# 번역
task_translated_text = translate_text(task1)
prompt_translated_text = translate_text(lazy_prompt1)
selected_text_translated_text = translate_text(selected_text)

print(task_translated_text)
print(prompt_translated_text)
print(selected_text_translated_text)

Summary

You are an expert in summarizing the content of a paper.
Your task is to write a professional summary.
Please write a summary that reflects the content of the selected_text list.

This study proposes a new treatment to improve glycemic control in people with diabetes. The new drug X reduced blood sugar variability by 30% compared to the existing drug and had a lower incidence of side effects.


In [20]:
print(create_chain("prompt_maker.yaml", task=task1, lazy_prompt=lazy_prompt1, selected_text=selected_text))

### Instruction

You are an expert in summarizing academic papers. Your task is to create a professional summary that accurately reflects the content of the paper.

### Context

Utilize the information provided in the selected_text list to craft your summary. 

### Requirements

- The summary should be concise yet comprehensive, capturing the main arguments and findings of the paper.
- Aim for a length of approximately 150-200 words.
- Use formal and academic language appropriate for a scholarly audience.
- Ensure that the summary is clear, coherent, and logically structured.


In [29]:
print(create_chain("prompt_maker.yaml", task=task_translated_text, lazy_prompt=prompt_translated_text, selected_text=selected_text_translated_text))

### Instructions for Summarizing a Paper

As an expert in academic summarization, your task is to craft a concise and professional summary of the content provided in the selected_text list. 

### Context and Requirements

- The summary should accurately reflect the main ideas, findings, and conclusions of the paper.
- Aim for a clear and concise format, ideally between 150-200 words.
- Use formal and academic language suitable for a scholarly audience.
- Highlight key points, methodologies, and any significant implications or contributions of the paper.
- Ensure the summary is self-contained, providing a comprehensive overview without requiring additional context.

Example:

"In this paper, the authors explore the impact of climate change on marine biodiversity, utilizing a combination of longitudinal data analysis and ecological modeling. The study reveals significant shifts in species distribution, emphasizing the urgent need for adaptive conservation strategies. Key findings include